# 04 — Series temporales

**Librerias usadas:** pandas, matplotlib

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'solineras'))
os.chdir(ROOT)
DATA = ROOT / 'data'

p = DATA / 'raw' / 'externas' / 'onei_anuario.csv'
if not p.exists():
    from extraer_onei import extraer
    extraer()
onei = pd.read_csv(p)
onei.head()

## 1. Generacion eolica + FV

In [ ]:
s = onei[onei['indicador'] == 'eolica_fotovoltaica_gwh'].sort_values('anio')
plt.figure(figsize=(10, 5))
plt.plot(s['anio'], s['valor'], marker='o', linewidth=2, color='darkorange')
plt.title('Generacion eolica + fotovoltaica (GWh)')
plt.xlabel('Anio')
plt.ylabel('GWh')
plt.grid(alpha=0.3)
plt.show()
print(s[['anio','valor']].to_string(index=False))

## 2. Potencia instalada

In [ ]:
s = onei[onei['indicador'] == 'eolica_fotovoltaica_mw'].sort_values('anio')
plt.figure(figsize=(10, 5))
plt.plot(s['anio'], s['valor'], marker='s', linewidth=2, color='steelblue')
plt.title('Potencia instalada PV + eolica (MW)')
plt.xlabel('Anio')
plt.ylabel('MW')
plt.grid(alpha=0.3)
plt.show()

## 3. CAGR

In [ ]:
def cagr(serie):
    if len(serie) < 2 or serie.iloc[0] == 0:
        return None
    v0, vn = serie.iloc[0], serie.iloc[-1]
    n = len(serie) - 1
    return ((vn / v0) ** (1/n) - 1) * 100

for ind in ['eolica_fotovoltaica_gwh', 'eolica_fotovoltaica_mw', 'total_gwh']:
    s = onei[onei['indicador'] == ind].sort_values('anio')['valor'].reset_index(drop=True)
    c = cagr(s)
    if c is not None:
        print(f'{ind:<30} CAGR = {c:+.2f}%')